In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config.config import MODELS_BASELINE

from src.features.build_features import build_features_dataframe, feature_columns 
from src.repository.parquet_repository import ParquetRepository

from src.models.AutoETSRegressor import AutoETSRegressor
from src.features.features import (sort_by_forecast_date, build_date_based_cv_splits)
from sklearn.model_selection import TimeSeriesSplit



d:\PROJETOS\git_repo\PharmaDemand-ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Exemplo futuro de leitura para o LightGBM, com h=15
repo = ParquetRepository(base_path=ROOT / "data")
df = repo.load("features/anomaly_predictions.parquet")

df_15 = df[(df["horizon_days"] == 7) |  (df["horizon_days"] == 15)]
X_train = df_15.loc[df_15["split"] == "train", feature_columns(df_15)]
y_train = df_15.loc[df_15["split"] == "train", "quantity"]
print(X_train.shape, y_train.shape)

[OK] Arquivo carregado: d:\PROJETOS\git_repo\PharmaDemand-ml\data\features\anomaly_predictions.parquet
(3712, 40) (3712,)


In [3]:
print (df.dtypes)

date                    datetime64[ns]
category                        object
horizon_days                     int64
forecast_origin         datetime64[ns]
split                           object
quantity                       float64
year                             int32
month                            int32
quarter                          int32
weekday                          int32
dayofyear                        int32
weekofyear                       int64
is_weekend                       int64
weekday_sin                    float64
weekday_cos                    float64
annual_sin                     float64
annual_cos                     float64
qty_lag_origin                 float64
qty_lag_origin_1               float64
qty_lag_origin_7               float64
qty_lag_origin_14              float64
qty_lag_origin_28              float64
qty_lag_origin_56              float64
qty_roll_mean_7                float64
qty_roll_std_7                 float64
qty_roll_sum_7           

In [4]:
X_test = df_15.loc[df_15["split"] == "test", feature_columns(df_15)]
y_test = df_15.loc[df_15["split"] == "test", "quantity"]
print(X_test.shape, y_test.shape)

(366, 40) (366,)


In [5]:
X_test.head()

,year,month,quarter,weekday,dayofyear,weekofyear,is_weekend,weekday_sin,weekday_cos,annual_sin,...,qty_roll_std_56,qty_mean_ratio_7_28,qty_mean_delta_7_28,qty_last_ratio_28,qty_last_zscore_28,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly
1860,2019,4,2,1,99,15,0,0.781831,0.623490,0.991269,...,11.555353,0.792710,-6.296429,0.790123,-0.575679,0.364572,0.0,0.856451,0.107515,0
1861,2019,4,2,2,100,15,0,0.974928,-0.222521,0.988854,...,11.684584,0.769873,-6.885714,0.611602,-1.028732,0.377550,0.0,0.837558,0.116822,0
1862,2019,4,2,3,101,15,0,0.433884,-0.900969,0.986146,...,11.632650,0.859483,-4.350000,1.259806,0.748972,0.346883,0.0,0.885301,0.107962,0
1863,2019,4,2,4,102,15,0,-0.433884,-0.900969,0.983147,...,11.377566,0.867529,-4.060714,0.822090,-0.506049,0.351566,0.0,0.861570,0.026718,0
1864,2019,4,2,5,103,15,1,-0.974928,-0.222521,0.979857,...,11.264431,0.877889,-3.603571,0.447295,-1.500537,0.368338,0.0,0.875710,0.143076,0


In [6]:
print (df_15.dtypes)

date                    datetime64[ns]
category                        object
horizon_days                     int64
forecast_origin         datetime64[ns]
split                           object
quantity                       float64
year                             int32
month                            int32
quarter                          int32
weekday                          int32
dayofyear                        int32
weekofyear                       int64
is_weekend                       int64
weekday_sin                    float64
weekday_cos                    float64
annual_sin                     float64
annual_cos                     float64
qty_lag_origin                 float64
qty_lag_origin_1               float64
qty_lag_origin_7               float64
qty_lag_origin_14              float64
qty_lag_origin_28              float64
qty_lag_origin_56              float64
qty_roll_mean_7                float64
qty_roll_std_7                 float64
qty_roll_sum_7           

In [7]:
#df_15["date"] = pd.to_datetime(df_15["date"])

df_train_model = df_15.loc[df_15["split"] == "train"].copy()
df_test_model = df_15.loc[df_15["split"] == "test"].copy()

df_train_model = df_train_model[(df_train_model["horizon_days"] == 15)]
df_test_model = df_test_model[(df_test_model["horizon_days"] == 15)]


print("Treinamento:", df_train_model.shape)
print("Teste:", df_test_model.shape)  

Treinamento: (1852, 46)
Teste: (183, 46)


In [8]:
print(df_15["split"].value_counts(dropna=False))

print("Treinamento:")
print(df_train_model["date"].min(), df_train_model["date"].max())

print("Teste:")
print(df_test_model["date"].min(), df_test_model["date"].max())

print("Categorias:")
print(df_15["category"].unique())

split
train    3712
test      366
Name: count, dtype: int64
Treinamento:
2014-03-14 00:00:00 2019-04-08 00:00:00
Teste:
2019-04-09 00:00:00 2019-10-08 00:00:00
Categorias:
['N02BE']


In [9]:
df_train_ets_clean = (
    df_train_model
    .sort_values("date")
    .copy()
)

serie = df_train_ets_clean["quantity"].astype(float)

# Zeros comprovadamente inválidos
invalidos = serie.eq(0)

# Ignorar zeros suspeitos no cálculo da mediana
historico = serie.mask(invalidos)

# Mediana dos 28 dias anteriores
substitutos = (
    historico.shift(1)
    .rolling(window=28, min_periods=1)
    .median()
)

df_train_ets_clean.loc[invalidos, "quantity"] = (
    substitutos.loc[invalidos]
)

if df_train_ets_clean["quantity"].isna().any():
    raise ValueError(
        "Existem zeros sem histórico anterior para imputação."
    )

In [10]:
LEAKAGE_SAFE_NUMERIC_FEATURES = [
    "qty_lag_origin", 
    "qty_lag_origin_7", 
    "qty_roll_mean_7",
    "qty_roll_std_7", 
    "qty_roll_mean_28", 
    "qty_roll_std_28",
    "qty_mean_ratio_7_28", 
    "qty_last_zscore_28", 
    "qty_zero_ratio_28",
    "weekday", 
    "annual_sin", 
    "annual_cos",
]

#LEAKAGE_SAFE_CATEGORICAL_FEATURES = ["Product", "Region"] nao tem

LAGGED_ANOMALY_FEATURES = [
    "anomaly_score",    
    "anomaly_score_mean7",
    "anomaly_score_mean15",
    "is_anomaly_rate7",
    "is_anomaly_rate7",
    "is_anomaly"
]

LEAKAGE_SAFE_HYBRID_NUMERIC_FEATURES = (
    LEAKAGE_SAFE_NUMERIC_FEATURES + LAGGED_ANOMALY_FEATURES
)

In [11]:
target_col = "quantity"

numeric_features = [
    "qty_lag_origin", 
    "qty_lag_origin_7", 
    "qty_roll_mean_7",
    "qty_roll_std_7", 
    "qty_roll_mean_28", 
    "qty_roll_std_28",
    "qty_mean_ratio_7_28", 
    "qty_last_zscore_28", 
    "qty_zero_ratio_28",
    "weekday", 
    "annual_sin", 
    "annual_cos",
]
#categorical_features = LEAKAGE_SAFE_CATEGORICAL_FEATURES.copy()



In [12]:
model = AutoETSRegressor(
    target_col=target_col,
    numeric_features=None,
    model_dir=MODELS_BASELINE,
    random_state=3,
    n_splits=3,
    scoring="neg_root_mean_squared_error",
    horizon=15,
    season_length=7,
    missing_dates="zero"
)

In [13]:
#print(model.calendar_diagnostics(df_train_model))
print(model.calendar_diagnostics(df_train_ets_clean))

{'date_start': '2014-03-14', 'date_end': '2019-04-08', 'existing_rows': 1852, 'expected_days': 1852, 'duplicate_dates': 0, 'missing_days': 0, 'missing_pct': 0.0, 'missing_dates_sample': []}


In [14]:
#grid_result = model.gridSearchFind(df_train_model)
grid_result = model.gridSearchFind(df_train_ets_clean)


print("Melhores parâmetros:", grid_result.best_params_)
print("RMSE CV:", -grid_result.best_score_)
print(f"RMSE CV: {grid_result.cv_results_}")

df_train_model = df_train_ets_clean

Melhores parâmetros: {'season_length': 1, 'model': 'ZZZ', 'damped': True}
RMSE CV: 12.09092004714355
RMSE CV: [{'params': {'season_length': 7, 'model': 'ZZZ', 'damped': None}, 'mean_test_score': -12.247086039599205, 'split_test_scores': [-12.861348958587618, -11.447118239732838, -12.43279092047716], 'error': None}, {'params': {'season_length': 7, 'model': 'ZZZ', 'damped': True}, 'mean_test_score': -12.154900179396977, 'split_test_scores': [-12.46581777078812, -11.91202324610643, -12.08685952129638], 'error': None}, {'params': {'season_length': 1, 'model': 'ZZZ', 'damped': None}, 'mean_test_score': -12.108439448153797, 'split_test_scores': [-12.775520786364023, -12.067080539553091, -11.482717018544278], 'error': None}, {'params': {'season_length': 1, 'model': 'ZZZ', 'damped': True}, 'mean_test_score': -12.09092004714355, 'split_test_scores': [-12.658001524880245, -12.150683630982755, -11.464074985567649], 'error': None}]


In [15]:
import pandas as pd

# Primeira data após o treinamento
data_inicio = pd.Timestamp(model.last_train_date) + pd.Timedelta(days=1)
data_fim = data_inicio + pd.Timedelta(days=14)

# CORREÇÃO: utilizar o dataset de TESTE
df_train_model = df_test_model.copy()

df_train_model["date"] = pd.to_datetime(df_train_model["date"])

df_test_ets = df_train_model.loc[
    df_train_model["date"].between(data_inicio, data_fim)
].sort_values("date").copy()

print("Período:", data_inicio, "até", data_fim)
print("Quantidade de registros:", len(df_train_model))

metrics = model.evaluate(df_train_model)

print("Métricas:", metrics)

Período: 2019-04-09 00:00:00 até 2019-04-23 00:00:00
Quantidade de registros: 183
Métricas: {'mae': 8.177778141729972, 'rmse': 11.015070754493921, 'r2': -0.0004326983160318765, 'wape': 35.663445627663464}


C:\Users\AlexK\AppData\Local\Temp\ipykernel_45028\101563757.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  data_inicio = pd.Timestamp(model.last_train_date) + pd.Timedelta(days=1)
C:\Users\AlexK\AppData\Local\Temp\ipykernel_45028\101563757.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  data_fim = data_inicio + pd.Timedelta(days=14)


In [16]:
import pandas as pd

# Primeira data após o treinamento
data_inicio = pd.Timestamp(model.last_train_date) + pd.Timedelta(days=1)
data_fim = data_inicio + pd.Timedelta(days=14)

# CORREÇÃO: utilizar o dataset de TESTE
df_test_ets = df_test_model.copy()

df_test_ets["date"] = pd.to_datetime(df_test_ets["date"])

df_test_ets = df_test_ets.loc[
    df_test_ets["date"].between(data_inicio, data_fim)
].sort_values("date").copy()

print("Período:", data_inicio, "até", data_fim)
print("Quantidade de registros:", len(df_test_ets))

metrics = model.evaluate(df_test_ets)

print("Métricas:", metrics)

Período: 2019-04-09 00:00:00 até 2019-04-23 00:00:00
Quantidade de registros: 15
Métricas: {'mae': 8.873272111231826, 'rmse': 11.75865608788218, 'r2': -0.0648419851491282, 'wape': 43.95610358932544}


C:\Users\AlexK\AppData\Local\Temp\ipykernel_45028\3502880299.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  data_inicio = pd.Timestamp(model.last_train_date) + pd.Timedelta(days=1)
C:\Users\AlexK\AppData\Local\Temp\ipykernel_45028\3502880299.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  data_fim = data_inicio + pd.Timedelta(days=14)


In [18]:
df_auditoria = df_15[["date", "split", "quantity"]].copy()

df_auditoria["is_zero"] = df_auditoria["quantity"].eq(0)

resumo = df_auditoria.groupby("split").agg(
    registros=("quantity", "size"),
    dias_zerados=("is_zero", "sum"),
    percentual_zeros=("is_zero", "mean")
)

resumo["percentual_zeros"] *= 100

display(resumo.round(2))

,registros,dias_zerados,percentual_zeros
split,,,
test,366,4,1.09
train,3712,46,1.24


In [19]:
df_zeros = df_auditoria.loc[
    df_auditoria["quantity"] == 0
].copy()

df_zeros["dia_semana"] = df_zeros["date"].dt.day_name()

display(
    df_zeros.groupby(["split", "dia_semana"])
    .size()
    .reset_index(name="quantidade")
)

,split,dia_semana,quantidade
0,test,Sunday,2
1,test,Thursday,2
2,train,Friday,4
3,train,Monday,10
4,train,Saturday,4
5,train,Sunday,12
6,train,Thursday,8
7,train,Tuesday,4
8,train,Wednesday,4
